# 49 · Logging and tracing an agent run
Level L5 · Part 6 · core · ~30 min · builds on 35

## Goal
Log every step of a Scout run as one JSON line, using OpenTelemetry's GenAI attribute names
(`gen_ai.request.model`, `gen_ai.usage.input_tokens`, ...), with a run id, the provider's response id,
cost, stop reason, tool and latency, and with emails and phone numbers redacted. Then read a bad run's
trace file and say what went wrong, which step was slowest and which cost the most.

## The idea
Lesson 35's loop prints a line per step. That is fine while you watch, useless at 3 a.m. when a user
says "Scout gave me nonsense". A **structured log** writes each step as a JSON object, one per line
(**JSON lines**), so a program can filter, sum and sort it. A **trace** is all the lines of one run,
tied together by a **run id**. **OpenTelemetry** (OTel) publishes shared names for these fields, the
**GenAI semantic conventions**, so any tracing tool can read them. Logs live longer than the run and are
read by more people, so personal data is **redacted** before a line is written.

Analogy: a flight recorder. It notes every action with the same fields, and investigators read it after
the landing went wrong.

## Picture
```mermaid
flowchart LR
    L["run_agent_logged"] -- "chat step" --> T["Tracer.event<br/>(adds run id, seq)"]
    L -- "tool step" --> T
    T --> R["redact<br/>emails, phones, secrets"]
    R --> F[("49_run.jsonl<br/>one JSON object per line")]
    F --> D["read back:<br/>slowest, most expensive,<br/>what went wrong"]
```

## Step by step
### 1. Setup: the recorder, the client and the price table
Every Gemini call is recorded once to `../data/llm_cassettes/49_logging-a-run.yaml` and replayed after
that, so the notebook runs with no key. Prices are lesson 12's table. Trace files go to `../data/traces/`.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("49_logging-a-run")

import ast, json, operator, os, re, time, uuid
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
TRACE_DIR = Path("../data/traces")
TRACE_DIR.mkdir(parents=True, exist_ok=True)
for old in TRACE_DIR.glob("49_*.jsonl"):
    old.unlink()
print("model:", MODEL, "| traces in:", TRACE_DIR)

LLM mode: replay (49_logging-a-run.yaml)


model: gemini-3.5-flash-lite | traces in: ../data/traces


The price table from lesson 12, plus a pretend price for the scripted fake, so the fake's runs have a
cost to log too.

In [2]:
PRICES_PER_MTOK = {            # USD per 1,000,000 tokens; source: Gemini pricing page, 2026-10-06
    "gemini-3.5-flash-lite": {"input": 0.30, "output": 2.50},
    "gemini-3.8-flash":      {"input": 0.75, "output": 3.75},
    "scripted-fake":         {"input": 0.30, "output": 2.50},   # pretend: priced like flash-lite
}

def cost_usd(model: str, input_tokens: int, output_tokens: int) -> float:
    price = PRICES_PER_MTOK[model]
    return (input_tokens * price["input"] + output_tokens * price["output"]) / 1_000_000

print(f"1,000 in + 200 out: ${cost_usd(MODEL, 1_000, 200):.5f}")

1,000 in + 200 out: $0.00080


### 2. Setup from lesson 35: Scout's notes, tools and `run_tool`
The six notes, the note search and the safe calculator (no `eval`), trimmed to what this lesson needs.

In [3]:
NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]

def search_notes(query: str) -> list[dict]:
    """Search the user's notes for a word or phrase (case-insensitive). Returns matching notes as {id, text}."""
    q = query.strip().lower()
    if not q:
        raise ValueError("query must not be empty, e.g. query='Mars'")
    return [n for n in NOTES if q in n["text"].lower()]

print(search_notes("day on Mars"))

[{'id': 3, 'text': 'A day on Mars lasts about 24 hours and 39 minutes.'}]


The calculator walks the syntax tree with an allow-list of operators, as in lesson 35.

In [4]:
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
       ast.Div: operator.truediv, ast.Pow: operator.pow}
HOW = "use numbers, + - * / ** and brackets, e.g. '24*60+39'"

def evaluate(node):
    if isinstance(node, ast.Constant) and type(node.value) in (int, float):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in OPS:
        return OPS[type(node.op)](evaluate(node.left), evaluate(node.right))
    raise ValueError(f"'{ast.unparse(node)[:40]}' is not plain arithmetic: {HOW}")

def calculator(expression: str) -> float:
    """Evaluate an arithmetic expression: numbers, + - * / ** and brackets. Use it for every calculation."""
    try:
        return evaluate(ast.parse(expression, mode="eval").body)
    except SyntaxError:
        raise ValueError(f"'{expression}' is not valid arithmetic: {HOW}") from None

print(calculator("24*60+39"))

1479


The tool table, the schemas Gemini needs, and lesson 35's `run_tool`, which never raises.

In [5]:
TOOLS = {"search_notes": search_notes, "calculator": calculator}
def one_arg(fn, arg): return {"name": fn.__name__, "description": fn.__doc__, "parameters": {
    "type": "object", "properties": {arg: {"type": "string"}}, "required": [arg]}}
SCHEMAS = [one_arg(search_notes, "query"), one_arg(calculator, "expression")]

def run_tool(call: dict, tools: dict) -> dict:
    """Run one tool call. Returns {"content": str, "is_error": bool}; never raises."""
    fn = tools.get(call["name"])
    if fn is None:
        return {"content": f"unknown tool '{call['name']}'. Available tools: {', '.join(tools)}.",
                "is_error": True}
    try:
        return {"content": json.dumps(fn(**call["args"])), "is_error": False}
    except Exception as e:                                    # any failure becomes an observation
        return {"content": f"{type(e).__name__}: {e}", "is_error": True}

print(run_tool({"name": "calculator", "args": {"expression": "24h*60"}}, TOOLS))

{'content': "ValueError: '24h*60' is not valid arithmetic: use numbers, + - * / ** and brackets, e.g. '24*60+39'", 'is_error': True}


### 3. A fake model that reports usage, and a fake clock
To log tokens, every model reply now carries a `usage` dict: model, response id, finish reason and token
counts. The scripted fake (lesson 35) makes up token counts as characters / 4 (lesson 03's rule of
thumb). A **fake clock** stands in for `time.perf_counter`: time moves only when told, so the fake
runs' latencies are the same every time.

In [6]:
class FakeClock:
    def __init__(self):
        self.now = 0.0
    def __call__(self) -> float:
        return self.now
    def advance(self, seconds: float) -> None:
        self.now += seconds

clock = FakeClock()
clock.advance(0.25)
print("fake time:", clock())

fake time: 0.25


The fake model: lesson 35's `ScriptedModel`, plus one fake second of thinking per call and a `usage`
dict on every reply.

In [7]:
class ScriptedModel:
    def __init__(self, script: list, clock: FakeClock, seconds: float = 0.8):
        self.script, self.clock, self.seconds, self.calls, self.next_id = script, clock, seconds, 0, 1

    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        self.calls += 1
        self.clock.advance(self.seconds)
        reply = json.loads(json.dumps(self.script[min(self.calls, len(self.script)) - 1]))
        for call in reply.get("tool_calls", []):
            call["id"], self.next_id = f"call_{self.next_id}", self.next_id + 1
        reply["usage"] = {"model": "scripted-fake", "response_id": f"fake-{self.calls}",
                          "finish_reason": "STOP", "input_tokens": len(json.dumps(messages)) // 4,
                          "output_tokens": len(json.dumps(reply)) // 4}
        return {"role": "assistant", **reply}

print(ScriptedModel([{"text": "hi"}], FakeClock())([{"role": "user", "text": "hello"}], []))

{'role': 'assistant', 'text': 'hi', 'usage': {'model': 'scripted-fake', 'response_id': 'fake-1', 'finish_reason': 'STOP', 'input_tokens': 8, 'output_tokens': 3}}


### 4. One log line: print versus JSON
The smallest example. Lesson 35 printed a sentence. A JSON line holds the same facts as named fields,
and `json.loads` gets them back exactly, so a program can filter on them.

In [8]:
step = {"step": 2, "tool": "calculator", "args": {"expression": "24*60+39"}, "ms": 3.0, "error": None}
print(f"step 2: calculator({step['args']}) -> 1479")         # lesson 35: for eyes only
line = json.dumps(step)                                        # a log line: for programs too
print(line)
back = json.loads(line)
print("read back:", back["tool"], back["ms"], "| is error:", back["error"] is not None)

step 2: calculator({'expression': '24*60+39'}) -> 1479
{"step": 2, "tool": "calculator", "args": {"expression": "24*60+39"}, "ms": 3.0, "error": null}
read back: calculator 3.0 | is error: False


### 5. The OpenTelemetry GenAI names
Instead of inventing `"tool"` and `"ms"`, use the published names (status "Development": they may still
change). A model call is operation `chat`, a tool call is `execute_tool`, the whole run is `invoke_agent`.
Names OTel does not define (run id, step, cost, latency) get our own prefix, `scout.`.

In [9]:
OTEL_NAMES = {
    "gen_ai.operation.name": "chat | execute_tool | invoke_agent",
    "gen_ai.provider.name": "gcp.vertex_ai",
    "gen_ai.request.model": "the model asked for",   "gen_ai.response.model": "the model that answered",
    "gen_ai.response.id": "the provider's id for this reply (a request id)",
    "gen_ai.response.finish_reasons": "why it stopped, e.g. ['STOP']",
    "gen_ai.usage.input_tokens": "tokens sent",       "gen_ai.usage.output_tokens": "tokens written",
    "gen_ai.tool.name": "which tool",                 "gen_ai.tool.call.id": "the model's id for the call",
    "gen_ai.agent.name": "which agent (Scout)",
    "error.type": "the error class, only when it failed",
}
for name, meaning in OTEL_NAMES.items():
    print(f"{name:32} {meaning}")

gen_ai.operation.name            chat | execute_tool | invoke_agent
gen_ai.provider.name             gcp.vertex_ai
gen_ai.request.model             the model asked for
gen_ai.response.model            the model that answered
gen_ai.response.id               the provider's id for this reply (a request id)
gen_ai.response.finish_reasons   why it stopped, e.g. ['STOP']
gen_ai.usage.input_tokens        tokens sent
gen_ai.usage.output_tokens       tokens written
gen_ai.tool.name                 which tool
gen_ai.tool.call.id              the model's id for the call
gen_ai.agent.name                which agent (Scout)
error.type                       the error class, only when it failed


### 6. Redact personal data before writing
OTel's rule: do not record prompts and outputs by default, because they hold user data. Scout logs the
question and tool arguments to make debugging possible, so it **redacts** them first. Each rule is a
pattern and a placeholder. A phone number must start with `+` or `0` and stand alone, so arithmetic such
as `24*60+39` or `1000 - 2000 - 3000` survives. Regex redaction is best-effort: it catches the common
shapes, not every way a person can write personal data, so treat it as a safety net, not a guarantee.

In [10]:
REDACTIONS = [
    (re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+"), "[EMAIL]"),
    (re.compile(r"(?<![\w.+-])(?:\+|0)\d[\d ()-]{7,}\d(?![\w-]|\.\d)"), "[PHONE]"),
    (re.compile(r"(?i)\bbearer\s+[\w.-]+|\bsk-[\w-]{20,}|\bAIza[\w-]{30,}"), "[SECRET]"),
]

def redact(value):
    if isinstance(value, dict):
        return {k: redact(v) for k, v in value.items()}
    if isinstance(value, list):
        return [redact(v) for v in value]
    if isinstance(value, str):
        for pattern, placeholder in REDACTIONS:
            value = pattern.sub(placeholder, value)
    return value

A test with made-up data: an `example.com` email, a UK drama phone number, a fake token built at run time.
The arithmetic must survive, and an email hidden under a key named like an id is still redacted.

In [11]:
fake_token = "Bearer " + "demo" * 6
sample = {"scout.question": "I'm ana.lima@example.com, +44 7700 900123. Mars day in minutes?",
          "scout.tool.args": {"expression": "24*60+39", "note": fake_token, "user.id": "a@b.com"},
          "scout.other_args": {"expression": "1000 - 2000 - 3000"}}
print(json.dumps(redact(sample), indent=1))

{
 "scout.question": "I'm [EMAIL], [PHONE]. Mars day in minutes?",
 "scout.tool.args": {
  "expression": "24*60+39",
  "note": "[SECRET]",
  "user.id": "[EMAIL]"
 },
 "scout.other_args": {
  "expression": "1000 - 2000 - 3000"
 }
}


### 7. A tracer: one JSON line per event
`Tracer` owns one run: its run id, a step counter `seq`, the clock and the file. `event` adds the run id
and `seq`, redacts, and appends one line. Only a fixed list of top-level id fields, `ID_KEYS`, skips
redaction: ids are opaque strings that can look like a phone number, and they must stay exact. The list is
exact names, never a pattern such as "ends with `.id`", because tool arguments are chosen by the model. A real run id is random (`uuid4`), so the notebook can pass a
fixed one to keep its output the same every run. No wall-clock timestamp, for the same reason; a
production log adds one.

In [12]:
ID_KEYS = {"gen_ai.response.id", "gen_ai.tool.call.id", "scout.run_id"}

class Tracer:
    def __init__(self, path: Path, run_id: str | None = None, clock=time.perf_counter):
        self.path, self.clock, self.seq = Path(path), clock, 0
        self.run_id = run_id or uuid.uuid4().hex[:16]
        self.path.write_text("")                             # one file per run

    def event(self, operation: str, **attrs) -> dict:
        self.seq += 1
        record = {"gen_ai.operation.name": operation, "scout.run_id": self.run_id,
                  "scout.seq": self.seq,
                  **{k: (v if k in ID_KEYS else redact(v)) for k, v in attrs.items()}}
        with self.path.open("a") as f:
            f.write(json.dumps(record) + "\n")
        return record

demo = Tracer(TRACE_DIR / "49_demo.jsonl", run_id="run-demo")
print(demo.event("invoke_agent", **{"gen_ai.agent.name": "Scout", "scout.question": sample["scout.question"]}))

{'gen_ai.operation.name': 'invoke_agent', 'scout.run_id': 'run-demo', 'scout.seq': 1, 'gen_ai.agent.name': 'Scout', 'scout.question': "I'm [EMAIL], [PHONE]. Mars day in minutes?"}


### 8. What to log for a model call and a tool call
Two helpers turn one step into one event. The chat event carries model, response id, finish reason,
tokens, cost and latency. The tool event carries the tool, the call id, the arguments, the result's
size (not its text) and, on failure, `error.type`.

In [13]:
def log_chat(tracer, step: int, reply: dict, ms: float) -> float:
    u = reply["usage"]
    cost = cost_usd(u["model"], u["input_tokens"], u["output_tokens"])
    tracer.event("chat", **{"scout.step": step, "gen_ai.provider.name": PROVIDER[u["model"]],
        "gen_ai.request.model": u["model"], "gen_ai.response.model": u.get("response_model", u["model"]),
        "gen_ai.response.id": u["response_id"], "gen_ai.response.finish_reasons": [u["finish_reason"]],
        "gen_ai.usage.input_tokens": u["input_tokens"], "gen_ai.usage.output_tokens": u["output_tokens"],
        "scout.cost_usd": round(cost, 8), "scout.duration_ms": round(ms, 1),
        "scout.tool_calls": [c["name"] for c in reply.get("tool_calls", [])]})
    return cost

def log_tool(tracer, step: int, call: dict, result: dict, ms: float) -> None:
    error = re.match(r"(\w+):", result["content"]) if result["is_error"] else None
    tracer.event("execute_tool", **{"scout.step": step, "gen_ai.tool.name": call["name"],
        "gen_ai.tool.call.id": call["id"], "scout.tool.args": call["args"],
        "scout.tool.result_chars": len(result["content"]), "scout.duration_ms": round(ms, 1),
        **({"error.type": error.group(1) if error else "tool_error"} if result["is_error"] else {})})

PROVIDER = {"scripted-fake": "scripted", MODEL: "gcp.vertex_ai"}
bad_call = {"name": "calculator", "id": "call_9", "args": {"expression": "24h*60"}}
log_tool(demo, 1, bad_call, run_tool(bad_call, TOOLS), 0.4)
print(demo.path.read_text().splitlines()[-1])

{"gen_ai.operation.name": "execute_tool", "scout.run_id": "run-demo", "scout.seq": 2, "scout.step": 1, "gen_ai.tool.name": "calculator", "gen_ai.tool.call.id": "call_9", "scout.tool.args": {"expression": "24h*60"}, "scout.tool.result_chars": 99, "scout.duration_ms": 0.4, "error.type": "ValueError"}


### 9. The loop, logged
Lesson 35's `run_agent`, with a timer around each model call and each tool call, an `invoke_agent` event
at the start and a `run_end` event with the stop reason, error count and total cost.

In [14]:
def run_agent_logged(question: str, model, tools: dict, schemas: list[dict], tracer: Tracer,
                     max_steps: int = 8) -> dict:
    clock, messages, errors, cost = tracer.clock, [{"role": "user", "text": question}], 0, 0.0
    tracer.event("invoke_agent", **{"gen_ai.agent.name": "Scout", "scout.question": question})
    for step in range(1, max_steps + 1):
        start = clock(); reply = model(messages, schemas)
        cost += log_chat(tracer, step, reply, (clock() - start) * 1000)
        messages.append(reply)
        if not reply.get("tool_calls"):
            tracer.event("run_end", **{"scout.stop": "end_of_turn", "scout.steps": step,
                         "scout.errors": errors, "scout.cost_usd": round(cost, 8), "scout.answer": reply["text"]})
            return {"answer": reply["text"], "stop": "end_of_turn", "steps": step, "errors": errors}
        for call in reply["tool_calls"]:
            start = clock(); result = run_tool(call, tools)
            log_tool(tracer, step, call, result, (clock() - start) * 1000)
            errors += result["is_error"]
            messages.append({"role": "tool", "call_id": call["id"], "name": call["name"], **result})
    tracer.event("run_end", **{"scout.stop": "step_limit", "scout.steps": max_steps,
                 "scout.errors": errors, "scout.cost_usd": round(cost, 8)})
    return {"answer": None, "stop": "step_limit", "steps": max_steps, "errors": errors}

### 10. A good run, traced
The happy path from lesson 35 on the fake, with the made-up email and phone number in the question.
`run_id` is fixed here only so the output stays the same.

In [15]:
QUESTION = "I'm ana.lima@example.com, +44 7700 900123. How many minutes long is a day on Mars?"
HAPPY = [{"tool_calls": [{"name": "search_notes", "args": {"query": "day on Mars"}}]},
         {"tool_calls": [{"name": "calculator", "args": {"expression": "24*60+39"}}]},
         {"text": "A day on Mars lasts about 1479 minutes."}]
clock = FakeClock()
good = Tracer(TRACE_DIR / "49_good.jsonl", run_id="run-good", clock=clock)
print(run_agent_logged(QUESTION, ScriptedModel(HAPPY, clock), TOOLS, SCHEMAS, good))
print(good.path.read_text().splitlines()[1])

{'answer': 'A day on Mars lasts about 1479 minutes.', 'stop': 'end_of_turn', 'steps': 3, 'errors': 0}
{"gen_ai.operation.name": "chat", "scout.run_id": "run-good", "scout.seq": 2, "scout.step": 1, "gen_ai.provider.name": "scripted", "gen_ai.request.model": "scripted-fake", "gen_ai.response.model": "scripted-fake", "gen_ai.response.id": "fake-1", "gen_ai.response.finish_reasons": ["STOP"], "gen_ai.usage.input_tokens": 28, "gen_ai.usage.output_tokens": 23, "scout.cost_usd": 6.59e-05, "scout.duration_ms": 800.0, "scout.tool_calls": ["search_notes"]}


That is one line of the file: a chat event. Reading raw JSON gets tiring, so `read_trace` loads a file
and `show` prints one row per event.

In [16]:
def read_trace(path: Path) -> list[dict]:
    return [json.loads(line) for line in Path(path).read_text().splitlines()]

def show(trace: list[dict], hide_ms: bool = False) -> None:
    for e in trace:
        what = e.get("gen_ai.tool.name") or e.get("gen_ai.request.model") or e.get("scout.stop", "")
        toks = (f"{e['gen_ai.usage.input_tokens']}+{e['gen_ai.usage.output_tokens']} tok"
                if "gen_ai.usage.input_tokens" in e else "")
        ms = "" if "scout.duration_ms" not in e else ("~ms" if hide_ms else f"{e['scout.duration_ms']} ms")
        print(f"{e['scout.seq']:>2} {e['gen_ai.operation.name']:13} {what:22} {toks:13} {ms:>9} "
              f"{e.get('error.type', '')}")

show(read_trace(good.path))

 1 invoke_agent                                                 
 2 chat          scripted-fake          28+23 tok      800.0 ms 
 3 execute_tool  search_notes                            0.0 ms 
 4 chat          scripted-fake          132+23 tok     800.0 ms 
 5 execute_tool  calculator                              0.0 ms 
 6 chat          scripted-fake          218+12 tok     800.0 ms 
 7 run_end       end_of_turn                                    


### 11. The slowest step and the most expensive step
Three lines of code each, on the file alone. The file has no email: the question was redacted.

In [17]:
trace = read_trace(good.path)
slowest = max((e for e in trace if "scout.duration_ms" in e), key=lambda e: e["scout.duration_ms"])
dearest = max((e for e in trace if "scout.cost_usd" in e and "scout.step" in e), key=lambda e: e["scout.cost_usd"])
print("slowest:", slowest["scout.seq"], slowest["gen_ai.operation.name"], slowest["scout.duration_ms"], "ms")
print("most expensive:", dearest["scout.seq"], f"${dearest['scout.cost_usd']:.6f}",
      dearest["gen_ai.usage.input_tokens"], "input tokens")
print("total cost:", trace[-1]["scout.cost_usd"], "| question logged as:", trace[0]["scout.question"])

slowest: 2 chat 800.0 ms
most expensive: 4 $0.000097 132 input tokens
total cost: 0.0002584 | question logged as: I'm [EMAIL], [PHONE]. How many minutes long is a day on Mars?


On the fake, every model call takes 800 ms and tools take 0 ms, so the first chat is "slowest" (ties go to
the first). Input tokens grow every call (28, 132, 218), because each call resends the whole history
(lesson 08). Yet the **second** chat costs most, not the last: output tokens cost over 8 times more
than input here, and the final answer was short. Only the logged numbers tell you that.

### 12. Debug a bad run from its transcript
A user reports: "Scout thought for ages and gave me nothing." Here is that run: a slow note search and a
model that keeps repeating itself. Pretend you were not there: you only get the file.

In [18]:
def slow(fn, seconds: float, clock: FakeClock):
    def wrapper(**args):
        clock.advance(seconds)                               # a slow backend, in fake time
        return fn(**args)
    return wrapper

clock = FakeClock()
SLOW_TOOLS = {"search_notes": slow(search_notes, 2.4, clock), "calculator": calculator}
BAD = [{"tool_calls": [{"name": "search_notes", "args": {"query": "Mars day length"}}]},
       {"tool_calls": [{"name": "search_notes", "args": {"query": "Mars day length"}}]},
       {"tool_calls": [{"name": "calculator", "args": {"expression": "24h*60+39"}}]},
       {"tool_calls": [{"name": "search_notes", "args": {"query": "Mars day length"}}]}]
bad = Tracer(TRACE_DIR / "49_bad.jsonl", run_id="run-bad", clock=clock)
run_agent_logged(QUESTION, ScriptedModel(BAD, clock), SLOW_TOOLS, SCHEMAS, bad, max_steps=4)
show(read_trace(bad.path))

 1 invoke_agent                                                 
 2 chat          scripted-fake          28+24 tok      800.0 ms 
 3 execute_tool  search_notes                         2400.0 ms 
 4 chat          scripted-fake          114+24 tok     800.0 ms 
 5 execute_tool  search_notes                         2400.0 ms 
 6 chat          scripted-fake          201+23 tok     800.0 ms 
 7 execute_tool  calculator                              0.0 ms ValueError
 8 chat          scripted-fake          311+24 tok     800.0 ms 
 9 execute_tool  search_notes                         2400.0 ms 
10 run_end       step_limit                                     


Now ask the file questions, in code: how did the run end, which calls repeated, what failed, and where
did the time go?

In [19]:
trace = read_trace(TRACE_DIR / "49_bad.jsonl")
end = trace[-1]
print("ended:", end["scout.stop"], "after", end["scout.steps"], "steps |", end["scout.errors"], "error(s)")
tools_run = [(e["gen_ai.tool.name"], json.dumps(e["scout.tool.args"])) for e in trace
             if e["gen_ai.operation.name"] == "execute_tool"]
print("repeated calls:", {c: tools_run.count(c) for c in set(tools_run) if tools_run.count(c) > 1})
for e in trace:
    if "error.type" in e or e.get("scout.tool.result_chars") == 2:     # "[]" is 2 characters
        print("seq", e["scout.seq"], e["gen_ai.tool.name"], e["scout.tool.args"],
              "->", e.get("error.type", "empty result"))
by_op = {}
for e in trace:
    by_op[e["gen_ai.operation.name"]] = by_op.get(e["gen_ai.operation.name"], 0) + e.get("scout.duration_ms", 0)
print("time by operation (ms):", by_op)

ended: step_limit after 4 steps | 1 error(s)
repeated calls: {('search_notes', '{"query": "Mars day length"}'): 3}
seq 3 search_notes {'query': 'Mars day length'} -> empty result
seq 5 search_notes {'query': 'Mars day length'} -> empty result
seq 7 calculator {'expression': '24h*60+39'} -> ValueError
seq 9 search_notes {'query': 'Mars day length'} -> empty result
time by operation (ms): {'invoke_agent': 0, 'chat': 3200.0, 'execute_tool': 7200.0, 'run_end': 0}


The diagnosis, from the file alone. The search for "Mars day length" returned an empty list (2 characters)
every time, because no note contains that exact phrase, and the model asked the same thing three times.
Its one calculator call used `24h`, a `ValueError`. The run hit the 4-step limit with no answer. The
three searches took 7200 ms of the 10400 ms: the slow backend, not the model. Two fixes follow: a search
that matches words, not one exact phrase, and a timeout on the search tool (lesson 30's per-call timeout).

### 13. Gemini behind the same loop
The `to_contents` converter from lesson 35, and a `GeminiModel` that also fills the `usage` dict from the
response: `response_id`, `model_version`, the finish reason and the token counts (thinking tokens are
billed as output, lesson 12).

In [20]:
def to_contents(messages: list[dict]) -> list[types.Content]:
    contents = []
    for m in messages:
        if m["role"] == "user":
            contents.append(types.Content(role="user", parts=[types.Part(text=m["text"])]))
        elif m["role"] == "assistant":
            contents.append(m["raw"])                          # keeps the thought signature
        else:
            response = {"error": m["content"]} if m["is_error"] else {"output": json.loads(m["content"])}
            part = types.Part(function_response=types.FunctionResponse(
                id=m["call_id"], name=m["name"], response=response))
            if contents[-1].role == "user" and contents[-1].parts[0].function_response:
                contents[-1].parts.append(part)
            else:
                contents.append(types.Content(role="user", parts=[part]))
    return contents

The wrapper. Temperature 0 and a short output limit keep the recording small.

In [21]:
SYSTEM = ("You are Scout. Answer questions about the user's notes. Search the notes first, with one or "
          "two words. Use the calculator for arithmetic. Answer in one sentence.")

class GeminiModel:
    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        decls = [types.Tool(function_declarations=[types.FunctionDeclaration(
            name=s["name"], description=s["description"], parameters_json_schema=s["parameters"]) for s in tools])]
        cfg = types.GenerateContentConfig(system_instruction=SYSTEM, temperature=0, max_output_tokens=300,
            tools=decls, automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))
        resp = client.models.generate_content(model=MODEL, contents=to_contents(messages), config=cfg)
        u, cand = resp.usage_metadata, resp.candidates[0]
        usage = {"model": MODEL, "response_model": resp.model_version, "response_id": resp.response_id,
                 "finish_reason": cand.finish_reason.name, "input_tokens": u.prompt_token_count or 0,
                 "output_tokens": (u.candidates_token_count or 0) + (u.thoughts_token_count or 0)}
        calls = [{"id": fc.id, "name": fc.name, "args": dict(fc.args)} for fc in resp.function_calls or []]
        reply = {"tool_calls": calls} if calls else {"text": resp.text}
        return {"role": "assistant", **reply, "raw": cand.content, "usage": usage}

### 14. A real run, traced
A real clock this time (`time.perf_counter`) and a random run id, as in production. Both change on every
run, so `show` hides the latencies and the run id is printed only as its length. If the cell is skipped
(no key, no recording), `real` stays `None` and the next cells say so.

In [22]:
real = None
print("real run will be written to", TRACE_DIR / "49_real.jsonl")

real run will be written to ../data/traces/49_real.jsonl


Scout on Gemini, same question with the same made-up email and phone number.

In [23]:
tracer = Tracer(TRACE_DIR / "49_real.jsonl")                  # real clock, random run id
real = run_agent_logged(QUESTION, GeminiModel(), TOOLS, SCHEMAS, tracer)
print(real)
print("run id length:", len(tracer.run_id))
assert all(e["scout.run_id"] == tracer.run_id for e in read_trace(tracer.path))   # id kept exact
show(read_trace(tracer.path), hide_ms=True)

{'answer': 'A day on Mars is 1,479 minutes long.', 'stop': 'end_of_turn', 'steps': 3, 'errors': 0}
run id length: 16
 1 invoke_agent                                                 
 2 chat          gemini-3.5-flash-lite  150+16 tok          ~ms 
 3 execute_tool  search_notes                               ~ms 
 4 chat          gemini-3.5-flash-lite  218+23 tok          ~ms 
 5 execute_tool  calculator                                 ~ms 
 6 chat          gemini-3.5-flash-lite  244+14 tok          ~ms 
 7 run_end       end_of_turn                                    


Gemini searched the notes, then used the calculator, then answered: three chat events and two tool
events, ended by `end_of_turn`. Now the same questions as before, on the real file.

In [24]:
if real is None:
    print("skipped: no real run (no key and no recording)")
else:
    trace = read_trace(TRACE_DIR / "49_real.jsonl")
    chats = [e for e in trace if e["gen_ai.operation.name"] == "chat"]
    dearest = max(chats, key=lambda e: e["scout.cost_usd"])
    print("finish reasons:", [e["gen_ai.response.finish_reasons"][0] for e in chats])
    print("response ids logged:", all(e["gen_ai.response.id"] for e in chats))
    print("most expensive: seq", dearest["scout.seq"], dearest["gen_ai.usage.input_tokens"], "in +",
          dearest["gen_ai.usage.output_tokens"], "out tokens", f"${dearest['scout.cost_usd']:.6f}")
    print(f"run total: ${trace[-1]['scout.cost_usd']:.6f} | answer: {trace[-1].get('scout.answer')}")

finish reasons: ['STOP', 'STOP', 'STOP']
response ids logged: True
most expensive: seq 4 218 in + 23 out tokens $0.000123
run total: $0.000316 | answer: A day on Mars is 1,479 minutes long.


### 15. Check the trace files hold no personal data and no secrets
Logs are copied to many places, so check them like code: search every trace file for the email, the phone
number and the usual names of credentials.

In [25]:
LEAKS = re.compile(r"(?i)authorization|x-goog-api-key|api_key|key=|bearer|ana\.lima|7700 900123")
for path in sorted(TRACE_DIR.glob("49_*.jsonl")):
    text = path.read_text()
    print(f"{path.name:16} {len(text.splitlines()):2} lines | leaks: {len(LEAKS.findall(text))} "
          f"| [EMAIL] placeholders: {text.count('[EMAIL]')}")

49_bad.jsonl     10 lines | leaks: 0 | [EMAIL] placeholders: 1
49_demo.jsonl     2 lines | leaks: 0 | [EMAIL] placeholders: 1
49_good.jsonl     7 lines | leaks: 0 | [EMAIL] placeholders: 1
49_real.jsonl     7 lines | leaks: 0 | [EMAIL] placeholders: 1


## At work
- **Teams log exactly these fields per step**: request id, model, token usage, latency, stop reason, tool
  name, arguments and result size, cost, all keyed by a run id (RESEARCH.md §7). Per-run cost in the log
  is what a budget is enforced against. → lesson 54
- **Prompts and outputs are off by default** in OTel's GenAI conventions; production keeps them in a
  separate, access-controlled store, referenced from the span. → lesson 50
- **Never log headers or keys.** A logged request header leaks the key to everyone who can read logs;
  this course's recorder strips them from cassettes for the same reason. → lesson 50
- **The GenAI names are still "Development"** and do change (`gen_ai.system` became
  `gen_ai.provider.name`), so teams map them in one place, like `log_chat` here.
- **Regex redaction is a safety net, not a guarantee.** Production also runs a DLP/PII scanner on the
  log pipeline. → lesson 50

## What just happened
- `Tracer` wrote one JSON object per line, each tagged with the run id and a sequence number.
- Chat events used OTel GenAI names for model, response id, finish reason and tokens; `scout.` names held
  cost and latency.
- `redact` replaced the email and phone number before any line was written; the leak check found none.
- On the good fake run, the slowest step was a model call, and the second call cost the most
  because output tokens are dearer than input.
- From the bad run's file alone we found the step limit, the repeated empty search, the `24h` error, and
  that the slow search took 7200 of 10400 ms.
- The real Gemini run produced the same kind of trace, with real response ids and token counts; there
  too the second call (218 in + 23 out tokens) cost the most.

## Common mistakes
**1. Logging sentences instead of JSON.** A printed line cannot be filtered or summed without fragile
parsing. Here a tool tries to read lesson 35's printed trace back.

In [26]:
printed = "step 2: calculator({'expression': '24*60+39'}) -> 1479"
json.loads(printed)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

Fix: write `json.dumps(record)` lines, as `Tracer.event` does; reading back is one `json.loads` per line.

**2. Logging the whole reply.** Gemini's reply dict holds `raw`, a `types.Content` object. `json.dumps`
cannot encode it, so the logger crashes the run it was meant to watch.

In [27]:
reply = {"role": "assistant", "text": "1479 minutes.",
         "raw": types.Content(role="model", parts=[types.Part(text="1479 minutes.")])}
json.dumps(reply)

TypeError: Object of type Content is not JSON serializable

Fix: log chosen fields (`log_chat` picks model, ids, tokens, cost), never the raw object.

**3. Writing before redacting.** A tracer that skips `redact` puts the user's email in the file.

In [28]:
raw_line = json.dumps({"scout.question": QUESTION})
assert "@" not in raw_line, f"personal data in the log: {raw_line}"

AssertionError: personal data in the log: {"scout.question": "I'm ana.lima@example.com, +44 7700 900123. How many minutes long is a day on Mars?"}

Fix: redact inside the one function that writes lines, so no caller can forget.

In [29]:
print(json.dumps(redact({"scout.question": QUESTION})))

{"scout.question": "I'm [EMAIL], [PHONE]. How many minutes long is a day on Mars?"}


## Production checklist
- [ ] Every model and tool call writes one JSON line with a run id and a sequence number.
- [ ] Chat lines carry model, response id, finish reason, input and output tokens, cost and latency.
- [ ] Tool lines carry tool name, call id, arguments, result size, latency and `error.type` on failure.
- [ ] Redaction runs inside the one function that writes, before anything reaches disk.
- [ ] Prompts and outputs are logged only on purpose, in a store with its own access control.
- [ ] No headers, keys or tokens are ever logged; a test greps the log files for them.
- [ ] Attribute names live in one place, so a change in the OTel conventions is a one-line fix.

## Try it
1. Add a redaction rule for payment card numbers such as `4111 1111 1111 1111` (placeholder `[CARD]`),
   and show that `redact` turns a question containing one into `[CARD]`, not `[PHONE]`.
2. Write `tool_report(path)` that reads a trace file and returns, per tool, the number of calls, the
   number of errors and the total milliseconds. On `49_bad.jsonl` it should show `search_notes` with 3
   calls and 7200.0 ms.

In [30]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [31]:
llm_replay.stop()

## Key terms
- **structured logging** — writing each event as named fields (here JSON) instead of a sentence, so programs can filter, sum and sort the log.
- **JSON lines** — a file with one JSON object per line; easy to append to and to read back line by line.
- **trace** — all the events of one run, tied together by its run id.
- **run id** — an id your code makes once per agent run and puts on every event of that run.
- **OpenTelemetry GenAI conventions** — shared attribute names for model and tool calls (`gen_ai.request.model`, `gen_ai.usage.input_tokens`, `gen_ai.tool.name`, ...), so any tracing tool can read them.
- **PII redaction** — replacing personal data (emails, phone numbers) and secrets with placeholders before a log line is written.
- **transcript debugging** — finding why a run went wrong by querying its trace file, not by rerunning it.